## KoBO Questionnaire Validator
Validates a KoBO (XLSForm) country questionnaire against a reference template.

Key differences from GeoPoll:
- Questions in **survey** sheet: `type`, `name`, `label::`, `relevant`, `required`, `constraint`, `Mandatory `
- Options in **choices** sheet: `list_name`, `name`, `label::`
- Skip logic = **relevant** column (XLSForm: `${var} = '1'`)
- Mandatory normalised from **Mandatory** column (yes / yes - panel / no)

In [89]:
import re
import polars as pl
import openpyxl
from pathlib import Path
from dataclasses import dataclass

## Step 1 — Configuration
Set your file paths and language here.

In [90]:
import yaml

# ── Load centralized config ───────────────────────────────────────────────────
# Edit validation_config.yaml — never touch this cell
_cfg_path = Path(__file__).parent / "validation_config.yaml" if "__file__" in dir() else \
            Path("validation_config.yaml")
cfg = yaml.safe_load(_cfg_path.read_text(encoding="utf-8"))["kobo"]

print(f"Config loaded from: {_cfg_path}")
print(f"  Country  : {Path(cfg['questionnaire_path']).name}")
print(f"  Language : {cfg['language']}")
print(f"  ISO3     : {cfg['iso3']}")
print(f"  Mode     : {cfg['reference_mode']}")
print(f"  Output   : {cfg.get('output_dir', '(same as questionnaire)')}")

Config loaded from: validation_config.yaml
  Country  : DIEM_Monitoring questionnaire_kobo_en_AFG_20260415.xlsx
  Language : en
  ISO3     : AFG
  Mode     : latest_template
  Output   : C:/Temp


## Step 2 — Reference file resolution

In [91]:
# Maps language code → label column name in KoBO XLSForms
LANG_LABEL_COL = {
    "en": "label::English (en)",
    "fr": "label::French (fr)",
    "ar": "label::Arabic (ar)",
    "es": "label::Spanish (es)",
}

def find_kobo_template(language: str, templates_dir: Path) -> Path:
    lang_upper = language.upper()
    candidates = sorted(
        templates_dir.glob(f"*kobo*{lang_upper}*template*.xlsx"),
        key=lambda p: p.stat().st_mtime, reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No KoBO template for language {language!r} in {templates_dir}")
    return candidates[0]

TEMPLATES_DIR = Path(cfg["templates_dir"])

if cfg["reference_mode"] == "latest_template":
    ref_path = find_kobo_template(cfg["language"], TEMPLATES_DIR)
elif cfg["reference_mode"] == "custom_reference":
    ref_path = Path(cfg["custom_reference_path"])
else:
    raise ValueError(f"Unknown reference_mode: {cfg['reference_mode']!r}")

run = {
    "questionnaire_path": cfg["questionnaire_path"],
    "reference_path"    : str(ref_path),
    "language"          : cfg["language"],
    "iso3"              : cfg["iso3"],
    "output_dir"        : cfg.get("output_dir") or str(Path(cfg["questionnaire_path"]).parent),
}

print(f"Reference : {ref_path.name}")
print(f"Country   : {Path(run['questionnaire_path']).name}")
print(f"Language  : {run['language']}   ISO3: {run['iso3']}")

Reference : household_questionnaire_kobo_EN_template_20250708_ISO3_F2F.xlsx
Country   : DIEM_Monitoring questionnaire_kobo_en_AFG_20260415.xlsx
Language  : en   ISO3: AFG


## Step 3 — Readers
`read_kobo_survey` — loads the **survey** sheet, normalises column names.  
`read_kobo_choices` — loads the **choices** sheet.  
`build_question_options` — joins them: one row per (Q Name, option).

In [92]:
_METADATA_TYPES = {
    "start", "end", "today", "deviceid", "phonenumber",
    "username", "simserial", "subscriberid", "background-audio",
}
# Types that can reference a choices list
_SELECT_TYPES = {"select_one", "select_multiple"}
# Real data-collection types (blank Mandatory → non-mandatory)
_DATA_TYPES = {
    "select_one", "select_multiple", "text", "integer", "decimal",
    "geopoint", "geotrace", "geoshape", "date", "time", "datetime",
    "file", "image", "audio", "video", "barcode", "range",
}

def _find_label_col(headers: list, language: str) -> str | None:
    target = LANG_LABEL_COL.get(language.lower(), f"label::{language}")
    if target in headers:
        return target
    return next((h for h in headers if h.startswith("label::")), None)


def _normalize_mandatory(val) -> str:
    v = str(val or "").strip().lower()
    if v in ("yes", "true", "mandatory"):
        return "mandatory"
    if v in ("yes - panel", "yes-panel", "mandatory-panel"):
        return "mandatory-panel"
    if v in ("no", "false"):
        return "non-mandatory"
    return ""   # blank for structural rows; caller upgrades to non-mandatory if needed


def read_kobo_survey(path: str, language: str = "en", _wb=None) -> pl.DataFrame:
    """
    Reads the survey sheet of a KoBO XLSForm.
    Output: Q Name, Q Type, list_name, label, required,
            mandatory_category, relevant, constraint, appearance, calculation,
            excel_row, source_file

    mandatory_category values:
      mandatory | mandatory-panel | non-mandatory | optional | "" (structural rows)
    """
    EMPTY = {
        "Q Name": pl.Utf8, "Q Type": pl.Utf8, "list_name": pl.Utf8,
        "label": pl.Utf8, "required": pl.Utf8, "mandatory_category": pl.Utf8,
        "relevant": pl.Utf8, "constraint": pl.Utf8, "appearance": pl.Utf8, "calculation": pl.Utf8,
        "excel_row": pl.Int64, "source_file": pl.Utf8,
    }
    owns_wb = _wb is None
    wb = _wb or openpyxl.load_workbook(path, data_only=True, read_only=True)
    try:
        ws   = wb["survey"]
        rows = list(ws.iter_rows(values_only=True))
    finally:
        if owns_wb:
            wb.close()

    if not rows:
        return pl.DataFrame(schema=EMPTY)

    headers   = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx   = {h: i for i, h in enumerate(headers)}
    label_col = _find_label_col(headers, language)
    mand_col  = next((h for h in headers if h.strip() == "Mandatory"), None)

    def get(row, col):
        i = col_idx.get(col)
        return row[i] if i is not None and i < len(row) else None

    records = []
    for excel_row, row in enumerate(rows[1:], start=2):
        q_type_raw = str(get(row, "type") or "").strip()
        q_name     = str(get(row, "name") or "").strip()
        if not q_name or not q_type_raw or q_type_raw in _METADATA_TYPES:
            continue

        parts     = q_type_raw.split(None, 1)
        q_type    = parts[0]
        list_name = parts[1].strip() if len(parts) > 1 and q_type in _SELECT_TYPES else None

        mand_cat = _normalize_mandatory(get(row, mand_col))
        if not mand_cat and q_type in _DATA_TYPES:
            mand_cat = "non-mandatory"
        if q_name.startswith("o_"):
            mand_cat = "optional"   # o_* = country optional module questions

        records.append({
            "Q Name"            : q_name,
            "Q Type"            : q_type,
            "list_name"         : list_name,
            "label"             : str(get(row, label_col) or "") if label_col else "",
            "required"          : str(get(row, "required") or "").strip().lower(),
            "mandatory_category": mand_cat,
            "relevant"          : str(get(row, "relevant") or "").strip(),
            "constraint"        : str(get(row, "constraint") or "").strip(),
            "appearance"        : str(get(row, "appearance") or "").strip(),
            "calculation"       : str(get(row, "calculation") or "").strip(),
            "excel_row"         : excel_row,
            "source_file"       : str(path),
        })
    return pl.DataFrame(records) if records else pl.DataFrame(schema=EMPTY)


def read_kobo_choices(path: str, language: str = "en", _wb=None) -> pl.DataFrame:
    """
    Reads the choices sheet.
    Output: list_name, option_name (str – the 'name' column), option_label
    option_name is the string key used for comparison (can be '1', 'dk', 'kabul', etc.)
    """
    EMPTY = {"list_name": pl.Utf8, "option_name": pl.Utf8, "option_label": pl.Utf8}
    owns_wb = _wb is None
    wb = _wb or openpyxl.load_workbook(path, data_only=True, read_only=True)
    try:
        ws   = wb["choices"]
        rows = list(ws.iter_rows(values_only=True))
    finally:
        if owns_wb:
            wb.close()
    if not rows:
        return pl.DataFrame(schema=EMPTY)

    headers   = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx   = {h: i for i, h in enumerate(headers)}
    label_col = _find_label_col(headers, language)

    def get(row, col):
        i = col_idx.get(col)
        return row[i] if i is not None and i < len(row) else None

    records = []
    for row in rows[1:]:
        list_name = get(row, "list_name")
        name_raw  = get(row, "name")
        if not list_name or name_raw is None:
            continue
        records.append({
            "list_name"   : str(list_name).strip(),
            "option_name" : str(name_raw).strip(),   # string key, always kept
            "option_label": str(get(row, label_col) or "").strip() if label_col else "",
        })
    return pl.DataFrame(records) if records else pl.DataFrame(schema=EMPTY)


def build_question_options(survey_df: pl.DataFrame, choices_df: pl.DataFrame) -> pl.DataFrame:
    """
    Join survey + choices to get one row per (Q Name, option).
    Only select_one / select_multiple questions have a choices list.
    Output: Q Name, Q Type, list_name, option_name, option_label, source_file
    """
    return (
        survey_df
        .filter(pl.col("Q Type").is_in(list(_SELECT_TYPES)))
        .filter(pl.col("list_name").is_not_null() & (pl.col("list_name") != ""))
        .select(["Q Name", "Q Type", "list_name", "source_file"])
        .join(choices_df, on="list_name", how="left")
        .select(["Q Name", "Q Type", "list_name", "option_name", "option_label", "source_file"])
    )


## Step 4 — Load files
Open each workbook once, pass it to both readers, then close.

In [93]:
current_wb   = openpyxl.load_workbook(run["questionnaire_path"], data_only=True, read_only=True)
reference_wb = openpyxl.load_workbook(run["reference_path"],     data_only=True, read_only=True)

current_survey    = read_kobo_survey(run["questionnaire_path"],  run["language"], _wb=current_wb)
current_choices   = read_kobo_choices(run["questionnaire_path"], run["language"], _wb=current_wb)
current_wb.close()

reference_survey  = read_kobo_survey(run["reference_path"],  run["language"], _wb=reference_wb)
reference_choices = read_kobo_choices(run["reference_path"], run["language"], _wb=reference_wb)
reference_wb.close()

current_options   = build_question_options(current_survey,   current_choices)
reference_options = build_question_options(reference_survey, reference_choices)

print(f"Current  : {current_survey.height} questions, {current_choices.height} choice rows")
print(f"Reference: {reference_survey.height} questions, {reference_choices.height} choice rows")
print()
print(current_survey.select(["Q Name", "Q Type", "mandatory_category", "relevant"]).head(12))


Current  : 335 questions, 1653 choice rows
Reference: 221 questions, 516 choice rows

shape: (12, 4)
┌────────────────┬─────────────┬────────────────────┬─────────────────────┐
│ Q Name         ┆ Q Type      ┆ mandatory_category ┆ relevant            │
│ ---            ┆ ---         ┆ ---                ┆ ---                 │
│ str            ┆ str         ┆ str                ┆ str                 │
╞════════════════╪═════════════╪════════════════════╪═════════════════════╡
│ audit          ┆ audit       ┆                    ┆                     │
│ audio_random   ┆ calculate   ┆                    ┆                     │
│ enum_team      ┆ select_one  ┆ non-mandatory      ┆                     │
│ enumerator     ┆ select_one  ┆ mandatory          ┆                     │
│ enumerator_    ┆ calculate   ┆ mandatory          ┆                     │
│ …              ┆ …           ┆ …                  ┆ …                   │
│ resp_agree     ┆ select_one  ┆ mandatory          ┆          

## Step 4b — Placeholder normalization

KoBO templates use `#placeholder#` tokens (e.g. `#ADMIN1#`, `#age#`, `#season#`) in label / constraint / option-label cells.
Country questionnaires have those tokens already replaced with real values.

**Before comparing**, we restore the country questionnaire's placeholder cells back to the template's vanilla form so the diff only flags genuine deviations — not the expected country-specific substitutions.
**After comparing**, the report shows the original filled-in values (not the tokens) via the `current_orig` parameter passed to each comparison function.

In [94]:
_PLACEHOLDER_RE = re.compile(r'#[^#]+#')


def read_additional_info(country_path: str, language: str = "en") -> dict[str, str]:
    """
    Read the country questionnaire's Additional information sheet.
    Layout: row 1 empty, row 2 = headers (Original | Replacement), rows 3+ = data.
    Returns {#placeholder#: actual_value} for every filled-in entry.
    """
    try:
        wb = openpyxl.load_workbook(country_path, data_only=True, read_only=True)
        if "Additional information" not in wb.sheetnames:
            wb.close()
            return {}
        ws   = wb["Additional information"]
        rows = list(ws.iter_rows(values_only=True))
        wb.close()
    except Exception as e:
        print(f"Warning: could not read Additional information sheet: {e}")
        return {}

    if len(rows) < 2:
        return {}
    headers = [str(h).strip() if h is not None else "" for h in rows[1]]

    orig_idx = next((i for i, h in enumerate(headers) if h.lower().startswith("original")), None)
    if language == "fr":
        repl_idx = next(
            (i for i, h in enumerate(headers) if "replacement" in h.lower() and "fr" in h.lower()), None
        )
    elif language == "ar":
        repl_idx = next(
            (i for i, h in enumerate(headers) if "replacement" in h.lower() and "ar" in h.lower()), None
        )
    else:
        repl_idx = next((i for i, h in enumerate(headers) if h.lower() == "replacement"), None)
        if repl_idx is None:
            repl_idx = next((i for i, h in enumerate(headers) if "replacement" in h.lower()), None)

    if orig_idx is None or repl_idx is None:
        print("Warning: Original/Replacement columns not found in Additional information sheet.")
        return {}

    pairs: dict[str, str] = {}
    for row in rows[2:]:
        orig = row[orig_idx] if orig_idx < len(row) else None
        repl = row[repl_idx] if repl_idx < len(row) else None
        if not orig or str(orig).strip() == "":
            continue
        repl_str = str(repl).strip() if repl is not None else ""
        if repl_str in ("", "nan", "None"):
            continue
        pairs[f"#{str(orig).strip()}#"] = repl_str
    return pairs


def restore_to_vanilla(
    current_df  : pl.DataFrame,
    reference_df: pl.DataFrame,
    cols        : list[str],
    key_cols    : list[str] | None = None,
) -> pl.DataFrame:
    """
    For every cell where the reference contains a #placeholder# token, overwrite
    the current questionnaire's value with the reference (vanilla) value.

    This prevents placeholder-filled country values from being flagged as deviations.
    key_cols: join keys — defaults to ["Q Name"]; use ["Q Name", "option_name"] for options.
    cols    : columns to restore (e.g. ["label", "constraint"] or ["option_label"]).
    """
    if key_cols is None:
        key_cols = ["Q Name"]
    result = current_df
    for col in cols:
        if col not in reference_df.columns or col not in current_df.columns:
            continue
        ref_ph = (
            reference_df
            .filter(pl.col(col).str.contains(r'#[^#]+#'))
            .select(key_cols + [col])
            .rename({col: "__ref_vanilla__"})
        )
        if ref_ph.height == 0:
            continue
        result = (
            result
            .join(ref_ph, on=key_cols, how="left")
            .with_columns(
                pl.when(pl.col("__ref_vanilla__").is_not_null())
                .then(pl.col("__ref_vanilla__"))
                .otherwise(pl.col(col))
                .alias(col)
            )
            .drop("__ref_vanilla__")
        )
    return result


def apply_replacements(
    df   : pl.DataFrame,
    pairs: dict[str, str],
    cols : list[str],
) -> pl.DataFrame:
    """Apply #placeholder# → actual_value replacements to specified columns (re-personalise after comparison)."""
    if not pairs:
        return df
    result = df
    for col in cols:
        if col not in result.columns:
            continue
        expr = pl.col(col)
        for key, val in pairs.items():
            expr = expr.str.replace_all(re.escape(key), val, literal=True)
        result = result.with_columns(expr.alias(col))
    return result

In [95]:
_VANILLA_COLS_SURVEY  = ["label", "constraint"]
_VANILLA_COLS_OPTIONS = ["option_label"]

# Read placeholder replacements from country questionnaire's Additional information sheet
replacement_pairs = read_additional_info(run["questionnaire_path"], run["language"])
print(f"Loaded {len(replacement_pairs)} placeholder pair(s) from Additional information sheet:")
for k, v in replacement_pairs.items():
    print(f"  {k!r}  →  {v!r}")

# Restore: for cells where the template has #placeholder#, copy template value → current
# This makes both sides identical for those cells so comparison won't flag them
current_survey_vanilla  = restore_to_vanilla(
    current_survey,  reference_survey,  _VANILLA_COLS_SURVEY
)
current_options_vanilla = restore_to_vanilla(
    current_options, reference_options, _VANILLA_COLS_OPTIONS,
    key_cols=["Q Name", "option_name"],
)

_n_survey  = sum(
    reference_survey.filter(pl.col(c).str.contains(r'#[^#]+#')).height
    for c in _VANILLA_COLS_SURVEY if c in reference_survey.columns
)
_n_options = (
    reference_options.filter(pl.col("option_label").str.contains(r'#[^#]+#')).height
    if "option_label" in reference_options.columns else 0
)
print(f"\nVanilla restoration: {_n_survey} survey cell(s), {_n_options} option cell(s) restored to template tokens")

Loaded 17 placeholder pair(s) from Additional information sheet:
  '#phone number#'  →  '0700 700 7000'
  '#age#'  →  '18'
  '#ADMIN1#'  →  'province'
  '#ADMIN2#'  →  'district'
  '#reference year#'  →  'last year'
  '#season#'  →  'last Season'
  '#season phase#'  →  'Harvesting'
  '#local measurement unit#'  →  'Jerib'
  '#local measurement unit conversion#'  →  '0.2'
  '#currency#'  →  'Afghani'
  '#MIN AMOUNT#'  →  '370'
  '#THRESHOLD#'  →  '15000'
  '#local vegetables#'  →  'Okra, eggplant, lettuce,beans, potatoes, spinach, tomatoes, onions, carrot, cauliflower,'
  '#local fruits#'  →  'Apple, grapes, pomegranate, almond, raisins, fig, walnut'
  '#Expected percentage of respondents cultivating crops#'  →  '0.46'
  '#Expected percentage of respondents raising livestock#'  →  '0.34'
  '#Expected percentage of respondents involved in fishery#'  →  '0'

Vanilla restoration: 28 survey cell(s), 1 option cell(s) restored to template tokens


## Step 5 — Normalisation helpers

In [96]:
def normalize_text_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def normalize_simple_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def normalize_logic_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.replace_all(r"\s*([=<>!+\-*/(),])\s*", "$1")
        .str.strip_chars()
    )


## Step 6 — Comparison functions

| Function | Checks | Adapted for KoBO? |
|---|---|---|
| `compare_question_presence` | Questions added/removed | No change |
| `compare_mandatory_kobo` | `mandatory_category` changed | Adapted (no raw yes/no) |
| `compare_option_labels_single` | Option label text changed | No change |
| `compare_option_presence_single` | Options added/removed | No change |
| `validate_relevant` | `relevant` broken refs + changes | KoBO-specific |

In [97]:
def compare_question_presence(current, reference):
    """Returns (added, removed). Each row has is_optional flag."""
    key = "Q Name"
    added = (
        current.select([key])
        .join(reference.select([key]), on=key, how="anti")
        .with_columns(pl.col(key).str.starts_with("o_").alias("is_optional"))
    )
    removed = (
        reference.select([key])
        .join(current.select([key]), on=key, how="anti")
        .with_columns(pl.col(key).str.starts_with("o_").alias("is_optional"))
    )
    return added, removed


def compare_mandatory_kobo(current, reference):
    """Compare mandatory_category between current and reference."""
    EMPTY = {
        "issue_type": pl.Utf8, "Q Name": pl.Utf8, "field": pl.Utf8,
        "Mandatory": pl.Utf8, "Mandatory_ref": pl.Utf8, "excel_row": pl.Int64,
    }
    if "mandatory_category" not in current.columns or "mandatory_category" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    meaningful = {"mandatory", "mandatory-panel", "optional", "non-mandatory"}
    return (
        current.select(["Q Name", "mandatory_category", "excel_row"])
        .join(reference.select(["Q Name", "mandatory_category"]), on="Q Name", how="inner", suffix="_ref")
        .filter(
            pl.col("mandatory_category").is_in(list(meaningful)) |
            pl.col("mandatory_category_ref").is_in(list(meaningful))
        )
        .filter(pl.col("mandatory_category") != pl.col("mandatory_category_ref"))
        .with_columns([
            pl.lit("mandatory_column_mismatch").alias("issue_type"),
            pl.lit("mandatory_category").alias("field"),
        ])
        .rename({"mandatory_category": "Mandatory", "mandatory_category_ref": "Mandatory_ref"})
        .select(["issue_type", "Q Name", "field", "Mandatory", "Mandatory_ref", "excel_row"])
    )


def compare_list_name_changes(current, reference):
    """Flag questions where the choices list (list_name from type) changed."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    cur = current.filter(pl.col("list_name").is_not_null()).select(["Q Name", "list_name", "excel_row"])
    ref = reference.filter(pl.col("list_name").is_not_null()).select(["Q Name", "list_name"])
    result = (
        cur.join(ref, on="Q Name", how="inner", suffix="_ref")
        .filter(pl.col("list_name") != pl.col("list_name_ref"))
        .with_columns([
            pl.lit("choices_list_changed").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("type (list_name)").alias("field"),
            pl.col("list_name").alias("current"),
            pl.col("list_name_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )
    return result if result.height > 0 else pl.DataFrame(schema=EMPTY)


def compare_question_labels(current_vanilla, reference, current_orig=None):
    """
    Compare question label text.
    current_vanilla: current survey with #placeholder# tokens restored (no false positives).
    current_orig   : original current survey — used so the report shows the actual filled-in label.
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "label" not in current_vanilla.columns or "label" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current_vanilla.select(["Q Name", "label", "excel_row"])
        .join(reference.select(["Q Name", "label"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_text_expr("label").alias("_norm"),
            normalize_text_expr("label_ref").alias("_norm_ref"),
        ])
        .filter(pl.col("_norm") != pl.col("_norm_ref"))
        .filter(pl.col("label_ref") != "")
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    # Replace vanilla label with the actual filled-in label for display in the report
    if current_orig is not None and "label" in current_orig.columns:
        orig_labels = current_orig.select(["Q Name", "label"]).rename({"label": "_actual"})
        diff = (
            diff.join(orig_labels, on="Q Name", how="left")
            .with_columns(
                pl.when(pl.col("_actual").is_not_null())
                .then(pl.col("_actual"))
                .otherwise(pl.col("label"))
                .alias("label")
            )
            .drop("_actual")
        )

    return (
        diff
        .with_columns([
            pl.lit("label_mismatch").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("label").alias("field"),
            pl.col("label").alias("current"),
            pl.col("label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_constraint_changes(current_vanilla, reference, current_orig=None):
    """
    Compare constraint expressions.
    Uses vanilla current so #placeholder#-containing constraints are not flagged.
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "constraint" not in current_vanilla.columns or "constraint" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current_vanilla
        .select(["Q Name", "constraint", "excel_row"])
        .join(
            reference.select(["Q Name", "constraint"]),
            on="Q Name", how="inner", suffix="_ref",
        )
        .with_columns([
            normalize_logic_expr("constraint").alias("_norm"),
            normalize_logic_expr("constraint_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    if current_orig is not None and "constraint" in current_orig.columns:
        orig_c = current_orig.select(["Q Name", "constraint"]).rename({"constraint": "_actual"})
        diff = (
            diff.join(orig_c, on="Q Name", how="left")
            .with_columns(
                pl.when(pl.col("_actual").is_not_null())
                .then(pl.col("_actual"))
                .otherwise(pl.col("constraint"))
                .alias("constraint")
            )
            .drop("_actual")
        )

    return (
        diff
        .with_columns([
            pl.lit("constraint_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("constraint").alias("field"),
            pl.col("constraint").alias("current"),
            pl.col("constraint_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_required_changes(current, reference):
    """Compare required column with whitespace/case-tolerant normalization."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "required" not in current.columns or "required" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current.select(["Q Name", "required", "excel_row"])
        .join(reference.select(["Q Name", "required"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_simple_expr("required").alias("_norm"),
            normalize_simple_expr("required_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    return (
        diff
        .with_columns([
            pl.lit("required_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("required").alias("field"),
            pl.col("required").alias("current"),
            pl.col("required_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_appearance_changes(current, reference):
    """Compare appearance with expression normalization that ignores tiny spacing differences."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "appearance" not in current.columns or "appearance" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current.select(["Q Name", "appearance", "excel_row"])
        .join(reference.select(["Q Name", "appearance"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_logic_expr("appearance").alias("_norm"),
            normalize_logic_expr("appearance_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    return (
        diff
        .with_columns([
            pl.lit("appearance_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("appearance").alias("field"),
            pl.col("appearance").alias("current"),
            pl.col("appearance_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_calculation_changes(current, reference):
    """Compare calculation with expression normalization that ignores tiny spacing differences."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "calculation" not in current.columns or "calculation" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current.select(["Q Name", "calculation", "excel_row"])
        .join(reference.select(["Q Name", "calculation"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_logic_expr("calculation").alias("_norm"),
            normalize_logic_expr("calculation_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    return (
        diff
        .with_columns([
            pl.lit("calculation_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("calculation").alias("field"),
            pl.col("calculation").alias("current"),
            pl.col("calculation_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_option_labels_single(current_opts, reference_opts, lang_scope="EN"):
    """Option label text changes for matching Q Name + option_name."""
    return (
        current_opts
        .join(reference_opts, on=["Q Name", "option_name"], how="inner", suffix="_ref")
        .with_columns([
            normalize_text_expr("option_label").alias("_norm"),
            normalize_text_expr("option_label_ref").alias("_norm_ref"),
        ])
        .filter(pl.col("_norm") != pl.col("_norm_ref"))
        .drop(["_norm", "_norm_ref"])
        .with_columns([
            pl.lit("option_label_mismatch").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_name")]).alias("field"),
            pl.lit("medium").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "option_label_ref",
                 "severity", "excel_row", "lang_scope"])
    )


def compare_option_presence_single(current_opts, reference_opts, lang_scope="EN"):
    """Added / removed options for matching Q Name."""
    key_cols = ["Q Name", "option_name"]
    removed = (
        reference_opts.select(key_cols + ["option_label"])
        .join(current_opts.select(key_cols), on=key_cols, how="anti")
        .with_columns([
            pl.lit("removed_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_name")]).alias("field"),
            pl.lit("high").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "severity", "lang_scope"])
    )
    added = (
        current_opts.select(key_cols + ["option_label"])
        .join(reference_opts.select(key_cols), on=key_cols, how="anti")
        .with_columns([
            pl.lit("added_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_name")]).alias("field"),
            pl.lit("medium").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "severity", "lang_scope"])
    )
    return added, removed

## Step 7 — Relevant validation (KoBO skip logic)

Two layers:
1. **Broken reference** (high): `${var}` in `relevant` points to a Q Name not in the current survey
2. **Relevant modified** (medium): expression differs from template — may need review

In [98]:
_KOBO_REF_RE = re.compile(r'\$\{([^}]+)\}')


def validate_relevant(
    current_survey : pl.DataFrame,
    reference_survey: pl.DataFrame,
) -> pl.DataFrame:
    """
    Layer 1 – broken ${var} references in relevant (high)
    Layer 2 – relevant expression changed from template (medium)
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }

    curr_qnames = set(current_survey["Q Name"].to_list())

    cur_rows = {r["Q Name"]: r for r in
                current_survey.select(["Q Name", "relevant", "excel_row"]).iter_rows(named=True)}
    ref_rows = {r["Q Name"]: r for r in
                reference_survey.select(["Q Name", "relevant"]).iter_rows(named=True)}

    issues = []
    for qname, cr in cur_rows.items():
        relevant  = str(cr.get("relevant") or "").strip()
        excel_row = cr.get("excel_row")
        if not relevant:
            continue

        # Layer 1: broken ${var} references
        refs   = _KOBO_REF_RE.findall(relevant)
        broken = [v for v in refs if v not in curr_qnames]
        if broken:
            issues.append({
                "issue_type": "broken_relevant_reference",
                "set_name": "", "Q Name": qname, "field": "relevant",
                "current"  : relevant[:220],
                "reference": f"missing variables: {broken}",
                "severity" : "high", "excel_row": excel_row,
            })

        # Layer 2: changed from template
        ref_relevant = str(ref_rows.get(qname, {}).get("relevant") or "").strip()
        if ref_relevant and ref_relevant != relevant:
            issues.append({
                "issue_type": "relevant_modified",
                "set_name": "", "Q Name": qname, "field": "relevant",
                "current"  : relevant[:220],
                "reference": ref_relevant[:220],
                "severity" : "medium", "excel_row": excel_row,
            })

    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues).unique(
        subset=["issue_type", "Q Name", "field", "current", "reference"], keep="first"
    )


## Step 8 — Critical sets
Same logic as GeoPoll. Load `critical_sets.yaml` if present; otherwise skip.
Note: `mandatory_category` is passed directly — no extra normalisation needed.

In [99]:
import yaml
from pathlib import Path as _Path

_CRIT_YAML = _Path(
    "c:/Users/edoar/WORK/FAO/repo/questionnaire_validation_revision/scripts/critical_sets.yaml"
)
rules = {"exact_sets": {}, "min_count_sets": {}, "crop_harvest": {}}
if _CRIT_YAML.exists():
    with open(_CRIT_YAML, encoding="utf-8") as _f:
        rules = yaml.safe_load(_f) or rules
    print(f"Loaded critical_sets.yaml  exact_sets: {len(rules.get('exact_sets', {}))}")
else:
    print("No critical_sets.yaml found — critical-set checks will be skipped.")


def validate_critical_sets(questions_df, exact_sets):
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not exact_sets:
        return pl.DataFrame(schema=EMPTY)

    # KoBO uses mandatory_category (already normalised); add Mandatory alias for compat
    if "Mandatory" not in questions_df.columns and "mandatory_category" in questions_df.columns:
        questions_df = questions_df.with_columns(
            pl.col("mandatory_category").alias("Mandatory")
        )
    # Mandatory_norm: map to yes/no for comparison with yaml expected values
    questions_df = questions_df.with_columns(
        pl.when(pl.col("Mandatory").is_in(["mandatory", "yes"])).then(pl.lit("yes"))
        .when(pl.col("Mandatory").is_in(["mandatory-panel"])).then(pl.lit("yes"))
        .when(pl.col("Mandatory").is_in(["optional", "no"])).then(pl.lit("no"))
        .otherwise(pl.lit("")).alias("Mandatory_norm")
    )

    present_qnames = set(questions_df["Q Name"].to_list())
    issues = []
    for set_name, set_rules in exact_sets.items():
        required_names   = [r["q_name"] for r in set_rules if r.get("required", True)]
        present_in_set   = [r["q_name"] for r in set_rules if r["q_name"] in present_qnames]
        present_required = [q for q in required_names if q in present_qnames]

        if 0 < len(present_required) < len(required_names):
            issues.append({
                "issue_type": "partial_critical_set", "set_name": set_name, "Q Name": "",
                "field": "Q Name",
                "current"  : ", ".join(sorted(present_in_set)),
                "reference": f"Expected all {len(required_names)} required questions",
                "severity" : "high", "excel_row": None,
            })

        for rule in set_rules:
            q_name   = rule["q_name"]
            expected = rule.get("expected_mandatory", "")
            required = rule.get("required", True)
            if q_name not in present_qnames:
                issues.append({
                    "issue_type": "missing_critical_question" if required else "advisory_question",
                    "set_name": set_name, "Q Name": q_name,
                    "field": "Q Name", "current": "",
                    "reference": "present",
                    "severity": "high" if required else "medium", "excel_row": None,
                })
                continue
            if not expected:
                continue
            row = questions_df.filter(pl.col("Q Name") == q_name).select(
                ["Q Name", "Mandatory", "Mandatory_norm", "excel_row"]
            ).to_dicts()
            if not row:
                continue
            row = row[0]
            if row["Mandatory_norm"] != expected:
                issues.append({
                    "issue_type": "critical_mandatory_mismatch", "set_name": set_name,
                    "Q Name": q_name, "field": "Mandatory",
                    "current": row["Mandatory"], "reference": expected,
                    "severity": "high", "excel_row": row.get("excel_row"),
                })

    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues)


def validate_prefix_counts(questions_df, min_count_sets):
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not min_count_sets:
        return pl.DataFrame(schema=EMPTY)
    all_qnames = questions_df["Q Name"].to_list()
    issues = []
    for set_name, rule in min_count_sets.items():
        prefix    = rule.get("prefix", "")
        min_count = rule.get("min_count", 1)
        desc      = rule.get("description", f"At least {min_count} '{prefix}*' questions required")
        matched   = sorted(q for q in all_qnames if q.startswith(prefix))
        if len(matched) < min_count:
            found_str = f"{len(matched)} found" + (f": {', '.join(matched)}" if matched else " (none)")
            issues.append({
                "issue_type": "below_minimum_count", "set_name": set_name, "Q Name": "",
                "field": "count", "current": found_str, "reference": desc,
                "severity": "high", "excel_row": None,
            })
    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues)


Loaded critical_sets.yaml  exact_sets: 1


## Step 9 — Issue unifiers
Convert each comparison result to the common schema.

In [100]:
def make_presence_issues(added, removed):
    added_i = added.with_columns([
        pl.lit("added_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("present").alias("current"),
        pl.lit("missing_in_reference").alias("reference"),
        pl.lit("info").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_i = removed.with_columns([
        pl.lit("removed_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("missing_in_current").alias("current"),
        pl.lit("present").alias("reference"),
        pl.when(pl.col("is_optional")).then(pl.lit("info")).otherwise(pl.lit("high")).alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_i, removed_i], how="vertical")


def make_mandatory_issues(mandatory_diff):
    return (
        mandatory_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("Mandatory").alias("current"),
            pl.col("Mandatory_ref").alias("reference"),
            pl.lit("high").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_issues(option_diff):
    return (
        option_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("option_label").alias("current"),
            pl.col("option_label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_presence_issues(added_opts, removed_opts):
    added_i = added_opts.with_columns([
        pl.lit("").alias("set_name"),
        pl.col("option_label").alias("current"),
        pl.lit("(not in template)").alias("reference"),
        pl.lit("medium").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_i = removed_opts.with_columns([
        pl.lit("").alias("set_name"),
        pl.lit("(removed)").alias("current"),
        pl.col("option_label").alias("reference"),
        pl.lit("high").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_i, removed_i], how="vertical")


## Step 10 — Run pipeline

In [101]:
import time
_t0 = time.perf_counter()

added, removed = compare_question_presence(current_survey, reference_survey)
print(f"Questions added: {added.height}  removed: {removed.height}")

mandatory_diff = compare_mandatory_kobo(current_survey, reference_survey)
print(f"Mandatory mismatches: {mandatory_diff.height}")

list_name_issues = compare_list_name_changes(current_survey, reference_survey)
print(f"Choices-list changes: {list_name_issues.height}")

# Label and constraint — use vanilla versions (placeholder tokens restored)
label_issues      = compare_question_labels(current_survey_vanilla, reference_survey, current_survey)
required_issues   = compare_required_changes(current_survey_vanilla, reference_survey)
constraint_issues = compare_constraint_changes(current_survey_vanilla, reference_survey, current_survey)
appearance_issues = compare_appearance_changes(current_survey_vanilla, reference_survey)
calc_issues       = compare_calculation_changes(current_survey_vanilla, reference_survey)
print(f"Label mismatches: {label_issues.height}  Required changes: {required_issues.height}  Constraint changes: {constraint_issues.height}  Appearance changes: {appearance_issues.height}  Calculation changes: {calc_issues.height}")

# Options — exclude:
#   1. Country-specific list_names (admin from AGOL, crops from Crop list — not in template)
#   2. Questions whose choices list_name changed (already captured by list_name_issues)
_country_lists    = set(cfg.get("country_specific_list_names", []))
_list_changed_q   = set(list_name_issues["Q Name"].to_list())
_skip_opt_qnames  = set(cfg.get("exclude_qnames_from_option_compare", []))
if _skip_opt_qnames:
    print(f"Option-compare Q Name excludes: {sorted(_skip_opt_qnames)}")

cur_opts_cmp = (
    current_options_vanilla
    .filter(~pl.col("list_name").is_in(list(_country_lists)))
    .filter(~pl.col("Q Name").is_in(list(_list_changed_q)))
    .filter(~pl.col("Q Name").is_in(list(_skip_opt_qnames)))
)
ref_opts_cmp = (
    reference_options
    .filter(~pl.col("list_name").is_in(list(_country_lists)))
    .filter(~pl.col("Q Name").is_in(list(_list_changed_q)))
    .filter(~pl.col("Q Name").is_in(list(_skip_opt_qnames)))
)

option_diff              = compare_option_labels_single(cur_opts_cmp, ref_opts_cmp)
added_opts, removed_opts = compare_option_presence_single(cur_opts_cmp, ref_opts_cmp)
print(f"Option label changes: {option_diff.height}  added: {added_opts.height}  removed: {removed_opts.height}")

relevant_issues = validate_relevant(current_survey, reference_survey)
print(f"Relevant issues: {relevant_issues.height}")

critical_issues = validate_critical_sets(current_survey, rules.get("exact_sets", {}))
count_issues    = validate_prefix_counts(current_survey, rules.get("min_count_sets", {}))
print(f"Critical set issues: {critical_issues.height}  Count issues: {count_issues.height}")

# ── build common issues ────────────────────────────────────────────────────────
presence_issues     = make_presence_issues(added, removed)
mandatory_issues    = make_mandatory_issues(mandatory_diff)
option_label_issues = make_option_issues(option_diff)
option_pres_issues  = make_option_presence_issues(added_opts, removed_opts)

# Exclude option issues for questions not in both files
_excluded = set(added["Q Name"].to_list()) | set(removed["Q Name"].to_list())
if _excluded:
    option_label_issues = option_label_issues.filter(~pl.col("Q Name").is_in(list(_excluded)))
    option_pres_issues  = option_pres_issues.filter(~pl.col("Q Name").is_in(list(_excluded)))

all_issues = pl.concat(
    [presence_issues, mandatory_issues, list_name_issues,
     label_issues, required_issues, constraint_issues, appearance_issues, calc_issues,
     option_label_issues, option_pres_issues,
     critical_issues, count_issues, relevant_issues],
    how="vertical",
)

# Severity sort
all_issues = (
    all_issues
    .with_columns(
        pl.when(pl.col("severity") == "high").then(pl.lit(0))
        .when(pl.col("severity") == "medium").then(pl.lit(1))
        .otherwise(pl.lit(2))
        .alias("_s")
    )
    .sort(["_s", "issue_type", "Q Name"])
    .drop("_s")
)

# CS downgrade: if count minimum met, removed questions in that prefix → info
_passing_prefixes = []
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    if count_issues.filter(pl.col("set_name") == _set_name).height == 0:
        _prefix = _rule.get("prefix", "")
        if _prefix:
            _passing_prefixes.append(_prefix)

if _passing_prefixes:
    _exprs = [pl.col("Q Name").cast(pl.Utf8).str.starts_with(p) for p in _passing_prefixes]
    _q_matches = pl.any_horizontal(_exprs) if len(_exprs) > 1 else _exprs[0]
    all_issues = all_issues.with_columns(
        pl.when((pl.col("issue_type") == "removed_question") & _q_matches)
        .then(pl.lit("info"))
        .otherwise(pl.col("severity"))
        .alias("severity")
    )

# Attach mandatory_cat for export
_mand_lookup = (
    pl.concat([
        reference_survey.select(["Q Name", "mandatory_category"]).with_columns(pl.lit(0).alias("_p")),
        current_survey.select(["Q Name", "mandatory_category"]).with_columns(pl.lit(1).alias("_p")),
    ], how="vertical")
    .sort(["Q Name", "_p"])
    .group_by("Q Name")
    .agg(pl.col("mandatory_category").last().alias("mandatory_cat"))
)
all_issues = (
    all_issues
    .join(_mand_lookup, on="Q Name", how="left")
    .with_columns(pl.col("mandatory_cat").fill_null(""))
)

# Found info for Critical Sets sheet
_found_info = {}
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    _prefix  = _rule.get("prefix", "")
    _matched = sorted(q for q in current_survey["Q Name"].to_list() if q.startswith(_prefix))
    _found_info[_set_name] = _matched

print(f"\nTotal issues: {all_issues.height}  ({time.perf_counter()-_t0:.2f}s)")
print(all_issues.select(["issue_type", "severity", "Q Name"]).head(15))

Questions added: 158  removed: 44
Mandatory mismatches: 0
Choices-list changes: 0
Label mismatches: 6  Required changes: 6  Constraint changes: 3  Appearance changes: 2  Calculation changes: 1
Option-compare Q Name excludes: ['enum_team', 'enumerator']
Option label changes: 0  added: 711  removed: 166
Relevant issues: 16
Critical set issues: 0  Count issues: 0

Total issues: 246  (0.06s)
shape: (15, 3)
┌───────────────────────────┬──────────┬───────────────────────────┐
│ issue_type                ┆ severity ┆ Q Name                    │
│ ---                       ┆ ---      ┆ ---                       │
│ str                       ┆ str      ┆ str                       │
╞═══════════════════════════╪══════════╪═══════════════════════════╡
│ broken_relevant_reference ┆ high     ┆ o_confl_agri              │
│ broken_relevant_reference ┆ high     ┆ o_confl_agri_otherspecify │
│ broken_relevant_reference ┆ high     ┆ o_confl_fs                │
│ broken_relevant_reference ┆ high     ┆ o

## Step 11 — Export

| Sheet | Content |
|---|---|
| **Summary** | Issue counts by category |
| **Critical Sets** | Structural validation (mandatory presence, count rules) |
| **Relevant Changes** | `relevant` broken references and modifications |
| **Question Changes** | Presence, mandatory flag, label text, constraint, choices list |
| **Option Changes** | Answer-option additions, removals, label changes |

In [102]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FILL_HIGH   = PatternFill("solid", fgColor="F4CCCC")
FILL_MEDIUM = PatternFill("solid", fgColor="FCE5CD")
FILL_INFO   = PatternFill("solid", fgColor="CFE2F3")
FILL_PASS   = PatternFill("solid", fgColor="D9EAD3")
FILL_HEADER = PatternFill("solid", fgColor="274E13")
FILL_SECT   = PatternFill("solid", fgColor="E8F5E9")

FONT_TITLE  = Font(bold=True, size=13, color="274E13")
FONT_HDR    = Font(bold=True, color="FFFFFF", size=11)
FONT_SECT   = Font(bold=True, color="274E13", size=11)
FONT_NORM   = Font(size=10)
THIN_BORDER = Border(
    left=Side(style="thin", color="CCCCCC"),
    right=Side(style="thin", color="CCCCCC"),
    bottom=Side(style="thin", color="CCCCCC"),
)
SEV_FILL = {"high": FILL_HIGH, "medium": FILL_MEDIUM, "info": FILL_INFO}

def _hdr(ws, row, vals):
    for c, v in enumerate(vals, 1):
        cell = ws.cell(row=row, column=c, value=v)
        cell.fill = FILL_HEADER; cell.font = FONT_HDR
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row].height = 18

def _sect(ws, row, title, n):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=n)
    c = ws.cell(row=row, column=1, value=title)
    c.fill = FILL_SECT; c.font = FONT_SECT
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20

def _style_row(ws, row, n, severity=""):
    fill = SEV_FILL.get(severity)
    for c in range(1, n + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = FONT_NORM; cell.border = THIN_BORDER
        cell.alignment = Alignment(vertical="top", wrap_text=True)
        if fill: cell.fill = fill

def _autofit(ws, mn=12, mx=60):
    for col_cells in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max(w+2, mn), mx)

COL_MAP = [
    ("issue_type",    "Issue type"),
    ("mandatory_cat", "Type"),
    ("set_name",      "Set"),
    ("Q Name",        "Q Name"),
    ("field",         "Field"),
    ("current",       "Current value"),
    ("reference",     "Reference / rule"),
    ("severity",      "Severity"),
    ("excel_row",     "Excel row"),
]

def _table(ws, start_row, df):
    cols = [(s, d) for s, d in COL_MAP if s in df.columns]
    _hdr(ws, start_row, [d for _, d in cols])
    r = start_row + 1
    if df.height == 0:
        ws.cell(row=r, column=1, value="No issues in this category").font = Font(bold=True, color="274E13", size=10)
        return r + 2
    for rd in df.to_dicts():
        for c, (src, _) in enumerate(cols, 1):
            ws.cell(row=r, column=c, value=rd.get(src))
        _style_row(ws, r, len(cols), rd.get("severity", ""))
        r += 1
    ws.freeze_panes = f"A{start_row+1}"
    ws.auto_filter.ref = f"A{start_row}:{get_column_letter(len(cols))}{start_row}"
    return r + 1


CRITICAL_ISSUE_TYPES = {
    "missing_critical_question", "advisory_question", "partial_critical_set",
    "critical_mandatory_mismatch", "below_minimum_count",
}
RELEVANT_ISSUE_TYPES  = {"broken_relevant_reference", "relevant_modified"}
QUESTION_CHANGE_TYPES = {
    "mandatory_column_mismatch", "added_question", "removed_question",
    "choices_list_changed", "label_mismatch", "required_modified",
    "constraint_modified", "appearance_modified", "calculation_modified",
}


def write_summary_sheet(wb, all_issues):
    ws = wb.create_sheet("Summary")
    ws.sheet_view.showGridLines = False
    for col, w in zip("ABCDE", [34, 8, 15, 15, 15]):
        ws.column_dimensions[col].width = w
    ws.merge_cells("A1:E1")
    ws["A1"] = "KoBO Questionnaire Validation Report"
    ws["A1"].font = FONT_TITLE
    ws["A1"].alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[1].height = 26

    r = 3
    _sect(ws, r, "ISSUE SUMMARY BY CATEGORY", 5); r += 1
    _hdr(ws, r, ["Category", "High", "Medium", "Info", "Total"]); r += 1
    cats = [
        ("Questions removed",       "removed_question"),
        ("Questions added",         "added_question"),
        ("Mandatory mismatches",    "mandatory_column_mismatch"),
        ("Label text changed",      "label_mismatch"),
        ("Required changed",        "required_modified"),
        ("Constraint changed",      "constraint_modified"),
        ("Appearance changed",      "appearance_modified"),
        ("Calculation changed",     "calculation_modified"),
        ("Choices list changed",    "choices_list_changed"),
        ("Options removed",         "removed_option"),
        ("Options added",           "added_option"),
        ("Option labels changed",   "option_label_mismatch"),
        ("Missing critical Q",      "missing_critical_question"),
        ("Count rule violations",   "below_minimum_count"),
        ("Broken relevant refs",    "broken_relevant_reference"),
        ("Relevant modified",       "relevant_modified"),
    ]
    for label, itype in cats:
        q    = all_issues.filter(pl.col("issue_type") == itype)
        high = q.filter(pl.col("severity") == "high").height
        med  = q.filter(pl.col("severity") == "medium").height
        info = q.filter(pl.col("severity") == "info").height
        ws.cell(row=r, column=1, value=label)
        ws.cell(row=r, column=2, value=high or "")
        ws.cell(row=r, column=3, value=med  or "")
        ws.cell(row=r, column=4, value=info or "")
        ws.cell(row=r, column=5, value=q.height or "")
        sev = "high" if high else ("medium" if med else "info" if info else "")
        _style_row(ws, r, 5, sev); r += 1


def write_critical_sets_sheet(wb, all_issues, found_info=None):
    ws = wb.create_sheet("Critical Sets")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(
        pl.col("issue_type").is_in(list(CRITICAL_ISSUE_TYPES))
    ).sort(["set_name", "severity", "Q Name"])
    _sect(ws, 1, "CRITICAL SETS  Structural validation issues", 8)
    next_row = _table(ws, 2, df)
    if found_info:
        next_row += 1
        _sect(ws, next_row, "QUESTIONS FOUND IN CURRENT QUESTIONNAIRE", 3)
        next_row += 1
        _hdr(ws, next_row, ["Set", "Count found", "Questions"])
        next_row += 1
        for set_name, questions in found_info.items():
            ws.cell(row=next_row, column=1, value=set_name)
            ws.cell(row=next_row, column=2, value=len(questions))
            ws.cell(row=next_row, column=3, value=", ".join(questions) if questions else "none found")
            _style_row(ws, next_row, 3, "info" if questions else "high")
            next_row += 1
    _autofit(ws)


def write_relevant_sheet(wb, all_issues):
    ws = wb.create_sheet("Relevant Changes")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(pl.col("issue_type").is_in(list(RELEVANT_ISSUE_TYPES)))
    _sect(ws, 1, "RELEVANT CHANGES  KoBO skip-logic (relevant column)", 8)
    _table(ws, 2, df)
    _autofit(ws)


def write_question_changes_sheet(wb, all_issues):
    ws = wb.create_sheet("Question Changes")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(pl.col("issue_type").is_in(list(QUESTION_CHANGE_TYPES)))
    _sect(ws, 1, "QUESTION CHANGES  Presence, mandatory, label, required, constraint, appearance, calculation, choices list", 8)
    _table(ws, 2, df)
    _autofit(ws)


def write_option_changes_sheet(wb, all_issues):
    ws = wb.create_sheet("Option Changes")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(pl.col("issue_type").is_in([
        "removed_option", "added_option", "option_label_mismatch",
    ]))
    _sect(ws, 1, "OPTION CHANGES  Answer options for questions in both files", 8)
    _table(ws, 2, df)
    _autofit(ws)


def export_report(all_issues, result_file, found_info=None):
    wb = Workbook()
    wb.remove(wb.active)
    write_summary_sheet(wb, all_issues)
    write_critical_sets_sheet(wb, all_issues, found_info=found_info)
    write_relevant_sheet(wb, all_issues)
    write_question_changes_sheet(wb, all_issues)
    write_option_changes_sheet(wb, all_issues)
    wb.save(result_file)
    print(f"Report saved: {result_file}")
    print(f"Sheets: {wb.sheetnames}")


# ── run export ─────────────────────────────────────────────────────────────────
from pathlib import Path as _P
_q_path = _P(run["questionnaire_path"])
_report_file = str(_q_path.parent / (_q_path.stem + "_validation_report.xlsx"))
export_report(all_issues, _report_file, found_info=_found_info)

Report saved: C:\Temp\DIEM_Monitoring questionnaire_kobo_en_AFG_20260415_validation_report.xlsx
Sheets: ['Summary', 'Critical Sets', 'Relevant Changes', 'Question Changes', 'Option Changes']


## Step 12 — Validated questionnaire output

Produces `validated_questionnaire_kobo_<lang>_<iso3>_<date>.xlsx` in `output_dir`:

1. Copy country questionnaire
2. Inject **crop choices** (crop / crop2 / crop3) from the `Crop list` sheet, selected first
3. Inject **admin choices** (admin1 / admin2) fetched live from FAO AGOL — no credentials needed
4. Restore template labels for `#placeholder#` questions (vanilla form before replacement)
5. Apply `#placeholder#` → actual values from `Additional information` sheet throughout survey + choices

In [103]:
import urllib.request as _urlreq
import json as _json
from shutil import copy2 as _copy2
from datetime import date as _date

# Special answer codes appended to each crop list
_CROP_SPECIALS = {
    "crop" : [("777", "No crop production"), ("888", "Don't know"), ("999", "Refused")],
    "crop2": [("666", "No other crop"),       ("888", "Don't know"), ("999", "Refused")],
    "crop3": [("666", "No other crop"),       ("888", "Don't know"), ("999", "Refused")],
}


def _read_crop_choices(src_path: str, language: str) -> dict[str, list[tuple]]:
    """
    Read 'Crop list' sheet → {list_name: [(code, label), ...]} for crop/crop2/crop3.
    Selected crops (marked in 'Select top 10 crops' col) are sorted first, then the rest.
    """
    wb = openpyxl.load_workbook(src_path, data_only=True, read_only=True)
    if "Crop list" not in wb.sheetnames:
        wb.close()
        return {}
    rows = list(wb["Crop list"].iter_rows(values_only=True))
    wb.close()

    # Layout: row 1 = title, row 2 = empty/sub-title, row 3 = header  (skiprows=2)
    if len(rows) < 3:
        return {}
    headers  = [str(h).strip() if h is not None else "" for h in rows[2]]
    sel_idx  = next((i for i, h in enumerate(headers) if "Select top" in h), None)
    code_idx = next((i for i, h in enumerate(headers) if "Dataset code" in h), None)
    if language == "fr":
        lbl_idx = next((i for i, h in enumerate(headers) if "Label" in h and "FR" in h), None)
    else:
        lbl_idx = next((i for i, h in enumerate(headers) if "Label" in h and "EN" in h), None)

    if code_idx is None or lbl_idx is None:
        return {}

    selected, unselected = [], []
    for row in rows[3:]:
        code = row[code_idx] if code_idx < len(row) else None
        lbl  = row[lbl_idx]  if lbl_idx  < len(row) else None
        sel  = row[sel_idx]  if sel_idx is not None and sel_idx < len(row) else None
        if code is None or str(code).strip() in ("", "nan"):
            continue
        entry = (str(code).strip(), str(lbl or "").strip())
        (selected if sel is not None and str(sel).strip() else unselected).append(entry)

    selected.sort(key=lambda x: x[0])
    unselected.sort(key=lambda x: x[0])
    combined = selected + unselected
    return {name: combined + specials for name, specials in _CROP_SPECIALS.items()}


def _fetch_admin_choices(iso3: str) -> dict[str, list[tuple]]:
    """
    Fetch adm1 and adm2 from public FAO AGOL REST service (no credentials needed).
    Returns:
      admin1: [(adm1_pcode, adm1_name), ...]
      admin2: [(adm2_pcode, adm2_name, adm1_pcode), ...]  ← adm1_pcode for choice_filter (col 5)
    """
    _BASE = ("https://services5.arcgis.com/sjP4Ugu5s0dZWLjd/arcgis/rest/services"
             "/Administrative_Boundaries_Reference_(view_layer)/FeatureServer")

    def _get(layer, fields):
        url = (f"{_BASE}/{layer}/query"
               f"?where=adm0_ISO3+%3D+%27{iso3}%27"
               f"&outFields={fields}&returnGeometry=false&outSR=4326&f=json")
        with _urlreq.urlopen(url, timeout=30) as r:
            return _json.loads(r.read().decode())["features"]

    adm1 = sorted(
        [(f["attributes"]["adm1_pcode"], f["attributes"]["adm1_name"])
         for f in _get(1, "adm1_name,adm1_pcode")],
        key=lambda x: x[0],
    )
    adm2 = sorted(
        [(f["attributes"]["adm2_pcode"], f["attributes"]["adm2_name"], f["attributes"]["adm1_pcode"])
         for f in _get(0, "adm2_name,adm2_pcode,adm1_pcode")],
        key=lambda x: x[0],
    )
    print(f"  AGOL  adm1: {len(adm1)} rows   adm2: {len(adm2)} rows")
    return {"admin1": adm1, "admin2": adm2}


def _rebuild_choices_sheet(wb, country_rows: dict[str, list[tuple]]) -> None:
    """
    Replace all rows belonging to country_rows list_names with fresh data.
    Non-country rows and empty separator rows are preserved in their original order.
    New country blocks are appended at the end, each preceded by an empty separator row.
    admin2 rows also populate col 5 (adm1_pcode) for choice_filter cascading.
    """
    ws       = wb["choices"]
    all_rows = list(ws.iter_rows(values_only=True))
    n_cols   = max((len(r) for r in all_rows), default=5)
    skip     = set(country_rows.keys())

    # Keep header + non-country rows + empty separator rows (r[0] is None)
    kept = [all_rows[0]] + [
        r for r in all_rows[1:]
        if r[0] is None or r[0] not in skip
    ]

    # Build new country blocks, each preceded by an empty separator row
    new_rows = []
    for list_name, entries in country_rows.items():
        new_rows.append([None] * n_cols)   # blank separator between blocks
        for entry in entries:
            row    = [None] * n_cols
            row[0] = list_name
            row[1] = entry[0]   # code / pcode
            row[2] = entry[1]   # label / name
            if len(entry) > 2 and n_cols >= 5:
                row[4] = entry[2]   # adm1_pcode in col 5 (choice_filter)
            new_rows.append(row)

    ws.delete_rows(1, ws.max_row)
    for r in kept + new_rows:
        ws.append(list(r))


def _apply_replacements_to_wb(wb, replacement_pairs: dict) -> None:
    """Apply #placeholder# → value to every string cell in survey and choices sheets."""
    if not replacement_pairs:
        return
    for sheet_name in ("survey", "choices"):
        if sheet_name not in wb.sheetnames:
            continue
        for row in wb[sheet_name].iter_rows():
            for cell in row:
                if isinstance(cell.value, str) and "#" in cell.value:
                    v = cell.value
                    for key, val in replacement_pairs.items():
                        if key in v:
                            v = v.replace(key, str(val))
                    cell.value = v


def produce_validated_questionnaire(
    cfg             : dict,
    reference_survey: pl.DataFrame,
    replacement_pairs: dict,
) -> str:
    """
    Build validated_questionnaire_kobo_<lang>_<iso3>_<date>.xlsx:
      1. Copy country questionnaire to output_dir
      2. Inject crop choices from 'Crop list' sheet
      3. Inject admin choices from AGOL
      4. Restore #placeholder# labels from template (then replace in step 5)
      5. Apply all #placeholder# replacements from Additional information sheet
    Returns the path to the saved file.
    """
    src  = cfg["questionnaire_path"]
    iso3 = cfg["iso3"]
    lang = cfg["language"]
    out  = Path(cfg.get("output_dir") or str(Path(src).parent))
    dest = str(out / f"validated_questionnaire_kobo_{lang}_{iso3}_{_date.today():%Y%m%d}.xlsx")

    print("Building validated questionnaire …")
    print(f"  Source : {Path(src).name}")
    print(f"  Output : {dest}")

    _copy2(src, dest)
    wb = openpyxl.load_workbook(dest)

    # ── 2. Crop choices ────────────────────────────────────────────────────────
    crop_rows = _read_crop_choices(src, lang)
    if crop_rows:
        print(f"  Crops  : {sum(len(v) for v in crop_rows.values())} entries  "
              f"({', '.join(crop_rows)})")

    # ── 3. Admin choices from AGOL ─────────────────────────────────────────────
    admin_rows: dict = {}
    try:
        admin_rows = _fetch_admin_choices(iso3)
    except Exception as e:
        print(f"  Warning: AGOL fetch failed — admin choices not updated  ({e})")

    country_rows = {**crop_rows, **admin_rows}
    if country_rows:
        _rebuild_choices_sheet(wb, country_rows)

    # ── 4. Restore template labels for #placeholder# questions ─────────────────
    #      Reset labels to canonical template form so step 5 substitutes correctly.
    if "survey" in wb.sheetnames and reference_survey.height > 0:
        ws_s      = wb["survey"]
        hdr       = [str(c.value or "").strip() for c in next(ws_s.iter_rows(min_row=1, max_row=1))]
        name_idx  = next((i for i, h in enumerate(hdr) if h == "name"), None)
        label_col = _find_label_col(hdr, lang)
        label_idx = hdr.index(label_col) if label_col and label_col in hdr else None

        if name_idx is not None and label_idx is not None:
            ref_ph = {
                r["Q Name"]: r["label"]
                for r in reference_survey
                    .filter(pl.col("label").str.contains(r"#[^#]+#"))
                    .select(["Q Name", "label"])
                    .iter_rows(named=True)
            }
            for row in ws_s.iter_rows(min_row=2):
                qn = str(row[name_idx].value or "").strip()
                if qn in ref_ph:
                    row[label_idx].value = ref_ph[qn]

    # ── 5. Apply #placeholder# → actual values ─────────────────────────────────
    _apply_replacements_to_wb(wb, replacement_pairs)

    wb.save(dest)
    print(f"  Saved  : {dest}")
    return dest


# ── Run ────────────────────────────────────────────────────────────────────────
_validated_path = produce_validated_questionnaire(cfg, reference_survey, replacement_pairs)


Building validated questionnaire …
  Source : DIEM_Monitoring questionnaire_kobo_en_AFG_20260415.xlsx
  Output : C:\Temp\validated_questionnaire_kobo_en_AFG_20260420.xlsx
  Crops  : 279 entries  (crop, crop2, crop3)
  AGOL  adm1: 34 rows   adm2: 399 rows
  Saved  : C:\Temp\validated_questionnaire_kobo_en_AFG_20260420.xlsx
